In [1]:
nombre_archivo = "Crazy Labs Apps – Privacy Policy.txt"

with open(nombre_archivo, "r") as archivo:
    texto = archivo.read()
    print("Contenido del archivo:")
    print(texto)

Contenido del archivo:
Crazy Labs Apps – Privacy Policy

Last Revised: May 5, 2021

Crazy Labs Ltd., its subsidiaries and affiliates (“Company” or “we””) respects the privacy of the users (“User” or “you”) of the mobile applications we distribute (“Apps”) and are committed to protect Users’ information. We believe you have a right to know our practices regarding information we may collect and use when you use our Apps.

IMPORTANT – we operate Apps for all ages:
(i) Apps distributed within our developer account under the brand “Crazy Labs” that do not include an age-gate and are intended for general audiences (“Crazy Labs Apps”), which are subject to our Crazy Labs Privacy Policy (defined below).

(ii) Apps distributed within our other developer accounts (except for Crazy Labs account) that do not include an age-gate and are treated as primarily directed at children (“Children Apps”), which are subject to our Children Privacy Policy (defined below).

(iii) Apps that include an age-gate 

In [4]:
import requests
from lxml import html
from bs4 import BeautifulSoup
import nltk
from fake_useragent import UserAgent

import re

In [8]:
nltk.download('words')
# nltk.download('punkt')

nlp = spacy.load("en_core_web_sm")

CACHE = 'https://webcache.googleusercontent.com/search?q=cache:'

# Input : Fragmento de las politicas de privacidad
# Output : Lista de oraciones
# Descripcion : Funcion que separa un texto extenso en oraciones

def separar_en_oraciones(texto):
    doc = nlp(texto)
    oraciones = [sent.text for sent in doc.sents]
    return oraciones

# Input : URL de las politicas de privacidad, que se encuentra en la Google Play Store
# Output : Contenido de la pagina (Completo)
# Descripcion : Funcion que checkea la respuesta de la pagina

def check_url(url):
    ua = UserAgent()
    HEADERS = {
        "User-Agent": ua.random,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Cache-Control": "max-age=0",
    }

    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        return response
    else:
        print("No se pudo obtener la pagina")
        print("Response : ", response)
        response = requests.get(CACHE + url, headers=HEADERS) # Se intenta con la version en cache
        return response


# Input : URL de la Google Play Store de la aplicacion
# Output : Lista con las oraciones de las politicas de privacidad
# Descripcion : Funcion obtiene las politicas de privacidad y limpia las oraciones
#               mediante expresiones regulares

def get_policies(url):
    response = requests.get(url, allow_redirects=True)
    tree = html.fromstring(response.content)
    app_url = tree.xpath('//*[@id="developer-contacts"]/div/div[4]/div/a')[0].get('href')
    app_response = check_url(app_url)
    soup = BeautifulSoup(app_response.content, 'html.parser')

    labels = ['a', 'script', 'style', 'option', 'input']

    for label in labels:
        for s in soup.select(label):
            s.extract()


    todas_oraciones = separar_en_oraciones(soup.get_text(separator=' | '))

    all_sentences = []
    for sentence in todas_oraciones:
        sentence = re.sub(r'[^\w\s]', '', sentence)
        sentence = re.sub(r'\w*[^a-zA-Z\s]+\w*', '', sentence)
        sentence = re.sub(r'^\s*$|\n|\t|\xa0|\r', '', sentence)
        sentence = sentence.strip(' ')
        sentence = re.split(r'\s{4,}', sentence)

        all_sentences.extend(sentence)

    return all_sentences

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


NameError: name 'spacy' is not defined